# Chapter 15: Instrumental Variables Estimation and Two Stage Least Squares

Topics: IV in simple models, additional exogenous regressors, 2SLS, exogeneity testing, overidentification.

## Step 1 – Setup: Working Directory & Output Folder

Run this cell first to fix the working directory and create the `output/` folder.

In [ ]:
import numpy as np
import pandas as pd
import wooldridge as woo
import matplotlib.pyplot as plt
import os
from linearmodels.iv import IV2SLS, IVLIML

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## 15.1 IV in Simple Regression

## Step 2 – The Problem of Endogeneity

**Endogeneity:** $\text{Cov}(x, u) \neq 0$, which causes OLS to be **biased and inconsistent**.

Common sources:
1. **Omitted variables** correlated with $x$
2. **Measurement error** in $x$
3. **Simultaneity** (reverse causation)

**Solution: Instrumental Variables (IV)**

An **instrument** $z$ must satisfy:
1. **Relevance:** $\text{Cov}(z, x) \neq 0$ (can be tested)
2. **Exogeneity:** $\text{Cov}(z, u) = 0$ (exclusion restriction — must be argued)

In [ ]:
df = woo.dataWoo('wage2').dropna()

# OLS (biased: educ endogenous)
ols = IV2SLS.from_formula('lwage ~ 1 + exper + [educ ~ educ]', data=df).fit()

# IV: instrument educ with sibs (siblings)
iv = IV2SLS.from_formula('lwage ~ 1 + exper + [educ ~ sibs]', data=df).fit()

print("OLS:")
print(ols.summary)
print("\nIV (instrument: sibs):")
print(iv.summary)

## 15.3 Two Stage Least Squares

## Step 3 – IV and 2SLS Estimation

**Simple IV estimator** (one instrument, one endogenous variable):
$$\hat{\beta}_1^{IV} = \frac{\text{Cov}(z, y)}{\text{Cov}(z, x)} = \frac{\sum_i(z_i-\bar{z})(y_i-\bar{y})}{\sum_i(z_i-\bar{z})(x_i-\bar{x})}$$

**Two-Stage Least Squares (2SLS):**

**Stage 1:** Regress endogenous $x$ on all instruments and exogenous variables:
$$x_i = \pi_0 + \pi_1 z_{1i} + \cdots + \pi_m z_{mi} + \mathbf{w}_i'\boldsymbol{\gamma} + v_i$$
→ obtain $\hat{x}_i$

**Stage 2:** Regress $y$ on $\hat{x}$ and exogenous variables:
$$y_i = \beta_0 + \beta_1 \hat{x}_i + \mathbf{w}_i'\boldsymbol{\delta} + u_i$$

**First-stage F-statistic:** instrument is weak if $F < 10$ (rule of thumb).

In [ ]:
df = woo.dataWoo('mroz').dropna()

# 2SLS: instrument educ with fatheduc and motheduc
iv2sls = IV2SLS.from_formula(
    'lwage ~ 1 + exper + expersq + [educ ~ fatheduc + motheduc]',
    data=df
).fit()
print("2SLS Results:")
print(iv2sls.summary)

## First Stage: Instrument Strength

## Step 4 – Overidentification Test (Sargan-Hansen J-test)

When there are **more instruments than endogenous variables** (overidentified):

**Sargan-Hansen J-statistic:**
$$J = n \cdot R^2_{\hat{u}} \sim \chi^2_{m-k}$$

where $\hat{u}$ are the 2SLS residuals regressed on all instruments, $m$ = number of instruments, $k$ = number of endogenous regressors.

H₀: all instruments are exogenous. Rejection casts doubt on instrument validity.

⚠️ The J-test cannot detect if **all** instruments are endogenous.

In [ ]:
import statsmodels.formula.api as smf

first_stage = smf.ols('educ ~ exper + expersq + fatheduc + motheduc', data=df).fit()
print(f"First Stage F-stat on instruments: {first_stage.fvalue:.2f}")
print(f"(Rule of thumb: F > 10 for strong instruments)")
print(first_stage.summary())

## 15.5 Testing Overidentifying Restrictions (Sargan Test)

In [ ]:
# Sargan-Hansen J-test: instruments valid?
print(iv2sls.wooldridge_overid)